# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata object
meta = dataset.metadata

# To print metadata, convert to a json serializable dict first
mdict = meta.to_json()
print(f"{mdict['name']}: {mdict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we'll print all available RecordSet `@id`s and their included fields (and each field's `@id`).

Note: Every entity is referenced by its `@id` as per Croissant best practices.

In [ ]:
# List all record sets and fields with their @id

record_sets = []

for recordset in meta.record_sets:
    print(f"RecordSet name: {recordset.name}, @id: {recordset.id}")
    record_sets.append(recordset.id)
    print("  Fields:")
    for field in recordset.fields:
        print(f"    - {field.name} (@id: {field.id}) type: {field.data_type}")
    print()

print("All available RecordSet @id values:")
for rsid in record_sets:
    print(f"- {rsid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview above.

In [ ]:
# If multiple record sets exist, you may select one for demonstration

# For this dataset, the main table is typically the first record set, but we print all to allow inspection.

# Fill in the chosen record set @id here:
if len(record_sets) == 0:
    print('No record sets found in this dataset.')
else:
    # Example: select the first record set
    main_record_set_id = record_sets[0]
    print(f"Selected RecordSet @id for extraction: {main_record_set_id}")

    # Optionally, override main_record_set_id with a specific @id if desired

    # Extract all record sets into DataFrames
    dataframes = {}
    for rsid in record_sets:
        print(f"Loading records for {rsid} ...")
        records = list(dataset.records(record_set=rsid))
        if len(records) > 0:
            dataframes[rsid] = pd.DataFrame(records)
            print(f"- Loaded {len(records)} records.")
        else:
            print(f"- No records found for {rsid}.")

    # Display columns and first 5 rows of the main record set
    if main_record_set_id in dataframes:
        print("Columns in main DataFrame:")
        print(dataframes[main_record_set_id].columns.tolist())
        display(dataframes[main_record_set_id].head())
    else:
        print(f"No data loaded for chosen RecordSet: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Typical operations include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

> **Remember:** Use Field `@id`s (as column names) for all data operations.

In [ ]:
# Choose numeric field and group field by @id (from the overview cell)
# "numeric_field_id" below should be replaced with the `@id` of a numeric field.
# For demonstration, we select the first numeric field in the record set (if available)

import numpy as np

if len(record_sets) > 0 and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Find the first numeric field (float or integer) from metadata
    main_rs = None
    for rs in meta.record_sets:
        if rs.id == main_record_set_id:
            main_rs = rs
            break

    numeric_field_id = None
    group_field_id = None
    if main_rs is not None:
        # Get first float or integer field
        for field in main_rs.fields:
            if str(field.data_type).lower().endswith('float') or str(field.data_type).lower().endswith('integer'):
                numeric_field_id = field.id
                break
        # Get first non-numeric field as group field
        for field in main_rs.fields:
            if not (str(field.data_type).lower().endswith('float') or str(field.data_type).lower().endswith('integer')):
                group_field_id = field.id
                break

    if numeric_field_id is not None and numeric_field_id in df.columns:
        print(f"Using numeric field: {numeric_field_id} (as column: '{numeric_field_id}')")
        # Attempt to coerce column to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 0

        # Filter numeric values over the mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optionally group by first non-numeric field
        if group_field_id is not None and group_field_id in df.columns:
            # Only group on finite values
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical/groupable field found for grouping.")
    else:
        print("No numeric field found for EDA!")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution (if selected)
if 'filtered_df' in locals() and numeric_field_id is not None and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
# Optionally, a boxplot by group field
if (
    'filtered_df' in locals() and
    group_field_id is not None and
    group_field_id in filtered_df.columns and
    numeric_field_id is not None and
    filtered_df[group_field_id].nunique() < 15  # Show only if not too many categories
):
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We explored the FAIR^2 mass spectrometry dataset using Croissant metadata and the `mlcroissant` library.
- Data structure, including record sets and fields, was inspected by their Croissant `@id` values.
- We performed basic EDA: filtered numeric records, normalized data, grouped by a categorical field, and visualized distributions.

Continue further analysis or modeling as needed for your scientific or project goals!